In [1]:
from datasets.fsmol_dock import FsDockDataset
from datasets.partitioned_fsmol_dock import FsDockDatasetPartitioned
# from visualize import make_fig



/home/alon.kitin/miniconda3/envs/FSdock/lib/python3.9/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/home/alon.kitin/miniconda3/envs/FSdock/lib/python3.9/site-packages/Bio/pairwise2.py:278: BiopythonDeprecationWarning: Bio.pairwise2 has been deprecated, and we intend to remove it in a future release of Biopython. As an alternative, please consider using Bio.Align.PairwiseAligner as a replacement, and contact the Biopython developers if you still need the Bio.pairwise2 module.
  warnings.warn(


In [14]:
import plotly.graph_objects as go

def plotly_edges(graph, start_l, end_l):
    start_pos = graph[start_l].pos
    end_pos = graph[end_l].pos
    edges = graph[start_l, end_l].edge_index
    xe=[]
    ye=[]
    ze=[]
    for s_pos, e_pos in zip(start_pos[edges[0]],end_pos[edges[1]]):
        xe+=[s_pos[0], e_pos[0], None]
        ye+=[s_pos[1], e_pos[1], None]
        ze+=[s_pos[2], e_pos[2], None]
    return {'x': xe, 'y':ye, 'z':ze, 'mode' : 'lines', 'name': f'{start_l}-{end_l}'}

def plotly_nodes(graph,l):
    poses = graph[l].pos
    xn=[]
    yn=[]
    zn=[]
    for pos in poses:
        xn.append(pos[0])
        yn.append(pos[1])
        zn.append(pos[2])
    return {'x': xn, 'y':yn, 'z':zn, 'mode': 'markers', 'name': l}

def plotly_holes(graph):
    poses = graph['ligand'].pos
    xn=[]
    yn=[]
    zn=[]
    for pos in poses[graph.hole_neighbors]:
        xn.append(pos[0])
        yn.append(pos[1])
        zn.append(pos[2])
    return {'x': xn, 'y':yn, 'z':zn, 'mode': 'markers', 'name': 'holes'}

def make_fig(graph) -> go.Figure:
    traces=[
        go.Scatter3d(**plotly_edges(graph, 'ligand', 'ligand'), line=dict(color='red', width=5)),
        go.Scatter3d(**plotly_nodes(graph, 'ligand'), marker=dict(symbol='circle', size=6, color='blue')),
        go.Scatter3d(**plotly_nodes(graph, 'receptor'), marker=dict(symbol='circle', size=3, color='green')),
        go.Scatter3d(**plotly_edges(graph, 'atom', 'receptor'), line=dict(color='yellow', width=3)),
        go.Scatter3d(**plotly_nodes(graph, 'atom'), marker=dict(symbol='circle', size=3, color='orange')),
        go.Scatter3d(**plotly_edges(graph, 'atom', 'atom'), line=dict(color='pink', width=1)),
        go.Scatter3d(**plotly_edges(graph, 'receptor', 'receptor'), line=dict(color='grey', width=3)),
        go.Scatter3d(**plotly_edges(graph, 'ligand', 'atom'), line=dict(color='light blue', width=3)),
        go.Scatter3d(**plotly_edges(graph, 'ligand', 'receptor'), line=dict(color='light green', width=3)),
        go.Scatter3d(**plotly_holes(graph), marker=dict(symbol='circle', size=7, color='black')),
        
    ]
    fig = go.Figure(data=traces, layout=go.Layout(
    width=1000,
    height=1000,
        ))
    return fig

In [15]:

import torch


def mask_graph_sidechains(graph, molecule_sidechain_mask_idx):
    device = graph['ligand'].x.device
    masks = {
        node_t:(
            (graph.sidechains_mask < molecule_sidechain_mask_idx).to(device)
            if node_t == "ligand"
            else torch.arange(graph[node_t].num_nodes, device=device)
        )
        for node_t in graph.metadata()[0]
    }
    return graph.subgraph(masks)


In [16]:

from datasets.fsmol_dock_clf import FsDockClfDataset

# ds = FsDockDatasetPartitioned('data/fsdock/smol','data/fsdock/smol_tasks.csv')
ds = FsDockDatasetPartitioned('data/fsdock/test','data/fsdock/test_tasks.csv', random_max_angle=1)
ds.load()

[2025-Apr-23 22:53:41 IDT] [partitioned_fsmol_dock.py:248] INFO - started load
[2025-Apr-23 22:54:44 IDT] [partitioned_fsmol_dock.py:267] INFO - finished load


In [17]:
# from datasets.process_mols import hide_sidechains

ds.random_max_angle=10
iterable = iter(range(100))


In [18]:
i = next(iterable)
i=100
graph = ds[i*10]
# print(ds._split_indexes[i])
make_fig(graph).show()

In [7]:
i = next(iterable)
i=100
graph = ds[i*10]
# print(ds._split_indexes[i])
make_fig(graph).show()

In [8]:
ei = {}
for s,d in zip(*graph['ligand','ligand'].edge_index):
    k = (s.item(),d.item())
    if k in ei:
        ei[k]+=1
    else:
        ei[k]=1
ei
        

{(0, 1): 2,
 (1, 0): 2,
 (1, 2): 2,
 (2, 1): 2,
 (2, 3): 2,
 (3, 2): 2,
 (3, 4): 2,
 (4, 3): 2,
 (3, 5): 2,
 (5, 3): 2,
 (5, 6): 2,
 (6, 5): 2,
 (6, 7): 2,
 (7, 6): 2,
 (7, 8): 2,
 (8, 7): 2,
 (8, 9): 2,
 (9, 8): 2,
 (9, 10): 2,
 (10, 9): 2,
 (10, 11): 2,
 (11, 10): 2,
 (2, 12): 2,
 (12, 2): 2,
 (12, 13): 2,
 (13, 12): 2,
 (13, 14): 2,
 (14, 13): 2,
 (14, 15): 2,
 (15, 14): 2,
 (15, 16): 2,
 (16, 15): 2,
 (16, 17): 2,
 (17, 16): 2,
 (17, 18): 2,
 (18, 17): 2,
 (17, 19): 2,
 (19, 17): 2,
 (19, 20): 2,
 (20, 19): 2,
 (20, 21): 2,
 (21, 20): 2,
 (13, 22): 2,
 (22, 13): 2,
 (22, 23): 2,
 (23, 22): 2,
 (23, 24): 2,
 (24, 23): 2,
 (24, 25): 2,
 (25, 24): 2,
 (25, 26): 2,
 (26, 25): 2,
 (26, 27): 2,
 (27, 26): 2,
 (26, 28): 2,
 (28, 26): 2,
 (28, 29): 2,
 (29, 28): 2,
 (22, 1): 2,
 (1, 22): 2,
 (29, 23): 2,
 (23, 29): 2,
 (11, 6): 2,
 (6, 11): 2,
 (20, 14): 2,
 (14, 20): 2,
 (13, 0): 1,
 (22, 0): 1,
 (23, 0): 1,
 (24, 0): 1,
 (29, 0): 1,
 (26, 0): 1,
 (27, 0): 1,
 (28, 0): 1,
 (25, 0): 1,
 (2

In [9]:
ea = graph['ligand','ligand'].edge_attr
eas = ea.sum(-1)
eas.sum()


tensor(66.)

In [10]:
from torch_geometric.transforms import ToUndirected

In [11]:
uea = ToUndirected()(graph)['ligand','ligand'].edge_attr
uea.shape

torch.Size([870, 4])

In [12]:
graph['ligand','ligand'].edge_attr.shape

torch.Size([936, 4])

In [13]:
510-48

462